# EventVAD smoke pipeline on Kaggle T4 × 2

Runs the bundled `Robbery137_x264.mp4` through video segmentation and frozen VideoLLaMA2 scoring, then merges per-segment anomaly scores. Select **GPU T4 × 2** and enable **Internet** in Kaggle settings. The model checkpoint is about 17 GB; allow enough scratch disk and download time.

This notebook uses short temporal windows as a dependable smoke-test segmenter. The repository’s graph/RAFT segmenter needs additional LAVIS and RAFT setup/checkpoint and is not invoked here, so this validates video ingestion and scoring, not the paper’s full segmentation method or benchmark metrics. Two independent scoring workers load one 4-bit model each, one per T4.

The notebook clones the public repository (or copies an attached `EventVAD` Kaggle dataset), then writes this workspace’s quantized inference runner and compatibility patch into its working copy. It fetches the bundled sample from GitHub if it is missing.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time, shutil, textwrap
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, f'{p.total_memory / 2**30:.1f} GiB')
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Select Kaggle GPU T4 × 2.'


In [ ]:
# Find an attached repo dataset, or clone the public upstream repository.
project = Path('/kaggle/input/eventvad-project/EventVAD')
if not project.exists():
    candidates = list(Path('/kaggle/input').glob('**/EventVAD/src/score/event_score.py'))
    if candidates:
        project = candidates[0].parents[2]
    else:
        project = Path('/kaggle/working/EventVAD')
        if not project.exists():
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/YihuaJerry/EventVAD.git', str(project)], check=True)

# Make the Kaggle input repo writable and provide this workspace's inference patches.
# Kaggle dataset mounts are read-only, so copy the lightweight source tree into /kaggle/working.
import json, shutil
source_project = project
working_project = Path('/kaggle/working/EventVAD')
if source_project.resolve() != working_project.resolve():
    if working_project.exists(): shutil.rmtree(working_project)
    shutil.copytree(source_project, working_project, ignore=shutil.ignore_patterns('.git', '.venv', 'models', 'outputs', '__pycache__'))
project = working_project
score_source = json.loads('"import os\\nimport csv\\nimport argparse\\nimport re\\nimport traceback\\nos.environ.setdefault(\\"PYTORCH_NO_CUDA_NVML\\", \\"1\\")\\nos.environ.setdefault(\\"PYTORCH_CUDA_ALLOC_CONF\\", \\"backend:cudaMallocAsync\\")\\nimport torch\\nimport torch.nn.functional as F\\nfrom pathlib import Path\\nfrom tqdm import tqdm\\nfrom videollama2 import model_init, mm_infer\\nfrom videollama2.utils import disable_torch_init\\n\\nNUM_FRAMES = 16\\nBATCH_SIZE = 4\\n\\ndef init_model(model_path, quantization=\\"4bit\\"):\\n    disable_torch_init()\\n    from transformers.models.siglip.modeling_siglip import SiglipAttention\\n\\n    # SigLIP 4.40 uses an eager fp32 attention matrix; SDPA computes the same\\n    # attention with much lower peak memory, which matters for 16 video frames.\\n    def siglip_sdpa_forward(self, hidden_states, attention_mask=None, output_attentions=False):\\n        batch, length, _ = hidden_states.shape\\n        shape = (batch, length, self.num_heads, self.head_dim)\\n        query = self.q_proj(hidden_states).view(shape).transpose(1, 2)\\n        key = self.k_proj(hidden_states).view(shape).transpose(1, 2)\\n        value = self.v_proj(hidden_states).view(shape).transpose(1, 2)\\n        output = F.scaled_dot_product_attention(\\n            query, key, value, attn_mask=attention_mask,\\n            dropout_p=self.dropout if self.training else 0.0,\\n            scale=self.scale,\\n        )\\n        output = output.transpose(1, 2).reshape(batch, length, self.embed_dim)\\n        return self.out_proj(output), None\\n\\n    SiglipAttention.forward = siglip_sdpa_forward\\n    if quantization == \\"4bit\\":\\n        load_args = {\\"device\\": \\"cuda:0\\", \\"load_4bit\\": True}\\n    elif quantization == \\"none\\":\\n        load_args = {\\"device\\": \\"cuda:0\\"}\\n    else:\\n        raise ValueError(f\\"Unsupported quantization mode: {quantization}\\")\\n    model, processor, tokenizer = model_init(model_path, **load_args)\\n    return model.eval(), processor, tokenizer\\n\\ndef parse_score(output):\\n    \\"\\"\\"Accept a bare scalar or a short answer containing one 0-1 score.\\"\\"\\"\\n    numbers = re.findall(r\\"(?<![\\\\w.])(?:0(?:\\\\.\\\\d+)?|1(?:\\\\.0+)?)(?![\\\\w.])\\", output)\\n    if not numbers:\\n        raise ValueError(f\\"Model did not return a score in [0, 1]: {output!r}\\")\\n    return float(numbers[-1])\\n\\ndef process_files(video_files, output_csv, model_path, quantization=\\"4bit\\"):\\n    model, processor, tokenizer = init_model(model_path, quantization)\\n\\n    abnormal_prompt = (\\n        \\"Assess whether this surveillance video segment contains anomalous or criminal activity. \\"\\n        \\"Return only a single anomaly score from 0 (normal) to 1 (clearly anomalous).\\"\\n    )\\n    \\n    details_csv = f\\"{output_csv}.details.csv\\"\\n    with open(details_csv, \'w\') as f:\\n        writer = csv.writer(f)\\n        writer.writerow([\'path\', \'score\', \'status\', \'response\'])\\n        \\n        for path_str in tqdm(video_files, desc=\\"Processing\\"):\\n            path = Path(path_str)\\n            if not path.exists():\\n                writer.writerow([str(path), -1, \\"Error: File not found\\", \\"\\"])\\n                continue\\n            try:\\n                video_input = processor[\'video\'](str(path))\\n                if torch.cuda.is_available():\\n                    video_input = video_input.to(\\"cuda:0\\")\\n                \\n                with torch.inference_mode():\\n                    output = mm_infer(\\n                        video_input,\\n                        abnormal_prompt,\\n                        model=model,\\n                        tokenizer=tokenizer,\\n                        do_sample=False,\\n                        max_new_tokens=32,\\n                        modal=\'video\'\\n                    )\\n                \\n                try:\\n                    score = parse_score(output)\\n                    status = \\"success\\"\\n                except ValueError as e:\\n                    score = -1\\n                    status = f\\"Error: {e}\\"\\n                \\n                writer.writerow([str(path), score, status, output])\\n            except Exception as e:\\n                writer.writerow([str(path), -1, f\\"Error: {e}\\\\n{traceback.format_exc()}\\", \\"\\"])\\n            finally:\\n                # empty_cache() can assert in CUDA MIG environments; normal\\n                # tensor lifetime management releases per-clip allocations.\\n                pass\\n\\ndef main(input_csv, output_csv, model_path, quantization=\\"4bit\\"):\\n    if not os.path.exists(input_csv):\\n        raise FileNotFoundError(f\\"Manifest\\u6587\\u4ef6\\u4e0d\\u5b58\\u5728: {input_csv}\\")\\n    \\n    output_dir = os.path.dirname(output_csv)\\n    os.makedirs(output_dir, exist_ok=True)\\n\\n    original_entries = []\\n    video_files = []\\n    with open(input_csv, \'r\') as f:\\n        for line in f:\\n            parts = line.strip().split()\\n            if len(parts) >= 3:\\n                path = parts[0]\\n                start = parts[1]\\n                end = parts[2]\\n                video_files.append(path)\\n                original_entries.append({\\n                    \'path\': path,\\n                    \'start\': start,\\n                    \'end\': end,\\n                    \'score\': None\\n                })\\n    \\n    process_files(video_files, output_csv, model_path, quantization)\\n    \\n    scores_dict = {}\\n    with open(f\\"{output_csv}.details.csv\\", \'r\') as f:\\n        reader = csv.reader(f)\\n        next(reader)\\n        for row in reader:\\n            path = row[0]\\n            score = row[1]\\n            scores_dict[path] = float(score) if row[2] == \\"success\\" else -1\\n\\n    with open(output_csv, \'w\') as f:\\n        for entry in original_entries:\\n            line = f\\"{entry[\'path\']} {entry[\'start\']} {entry[\'end\']} {scores_dict.get(entry[\'path\'], -1)}\\\\n\\"\\n            f.write(line)\\n\\nif __name__ == \\"__main__\\":\\n    import argparse\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--input_csv\\", required=True)\\n    parser.add_argument(\\"--output_csv\\", required=True)\\n    parser.add_argument(\\"--model\\", default=os.environ.get(\\n        \\"EVENTVAD_MODEL\\", \\"models/VideoLLaMA2.1-7B-16F\\"))\\n    parser.add_argument(\\"--quantization\\", choices=[\\"4bit\\", \\"none\\"], default=\\"4bit\\",\\n                        help=\\"Runtime quantization for the frozen checkpoint.\\")\\n    args = parser.parse_args()\\n    main(args.input_csv, args.output_csv, args.model,\\n         quantization=args.quantization)\\n\\n\\n\\n"')
patch_source = json.loads('"\\"\\"\\"Apply the Transformers 4.40 SigLIP attention compatibility fix upstream needs.\\"\\"\\"\\nimport argparse\\nfrom pathlib import Path\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"source\\", type=Path, help=\\"VideoLLaMA2 checkout or package directory\\")\\n    root = parser.parse_args().source\\n    encoder = root / \\"videollama2\\" / \\"model\\" / \\"encoder.py\\"\\n    source = encoder.read_text()\\n    old_single = \\"config._attn_implementation = \'flash_attention_2\'\\"\\n    old_double = \'config._attn_implementation = \\"flash_attention_2\\"\'\\n    if (old_single not in source and old_double not in source\\n            and \\"config._attn_implementation = \'eager\'\\" not in source\\n            and \'config._attn_implementation = \\"eager\\"\' not in source):\\n        raise SystemExit(f\\"Expected upstream attention setting not found in {encoder}\\")\\n    source = source.replace(old_single, \\"config._attn_implementation = \'eager\'\\")\\n    source = source.replace(old_double, \\"config._attn_implementation = \\\\\\"eager\\\\\\"\\")\\n    full_batch = \\"\\"\\"            image_forward_outs = self.vision_tower(images, output_hidden_states=True)\\n            image_features = self.feature_select(image_forward_outs)\\"\\"\\"\\n    chunked_batch = \\"\\"\\"            chunk_size = int(os.environ.get(\\"VIDEOLLAMA2_VISION_BATCH\\", \\"1\\"))\\n            if images.shape[0] > chunk_size:\\n                image_features = []\\n                for image_chunk in images.split(chunk_size):\\n                    image_forward_outs = self.vision_tower(image_chunk, output_hidden_states=True)\\n                    image_features.append(self.feature_select(image_forward_outs).to(images.dtype))\\n                image_features = torch.cat(image_features, dim=0)\\n            else:\\n                image_forward_outs = self.vision_tower(images, output_hidden_states=True)\\n                image_features = self.feature_select(image_forward_outs)\\"\\"\\"\\n    if full_batch in source:\\n        source = source.replace(full_batch, chunked_batch)\\n    encoder.write_text(source)\\n\\n    loader = root / \\"videollama2\\" / \\"model\\" / \\"__init__.py\\"\\n    source = loader.read_text()\\n    old_quant = \\"\\"\\"            bnb_4bit_quant_type=\'nf4\'\\n        )\\"\\"\\"\\n    new_quant = \\"\\"\\"            bnb_4bit_quant_type=\'nf4\',\\n            llm_int8_skip_modules=[\'vision_tower\', \'mm_projector\', \'lm_head\']\\n        )\\"\\"\\"\\n    if old_quant in source:\\n        source = source.replace(old_quant, new_quant, 1)\\n        loader.write_text(source)\\n    print(f\\"Patched SigLIP/CLIP vision attention to eager: {encoder}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"')
(project / 'src/score/event_score.py').write_text(score_source)
(project / 'scripts').mkdir(parents=True, exist_ok=True)
(project / 'scripts/patch_videollama2.py').write_text(patch_source)
video = project / 'src/event_seg/videos/Robbery137_x264.mp4'
if not video.exists():
    video.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/YihuaJerry/EventVAD/main/src/event_seg/videos/Robbery137_x264.mp4'
    urllib.request.urlretrieve(url, video)
print('Writable project:', project)
print('Sample:', video, f'({video.stat().st_size / 2**20:.1f} MiB)')


## Install scoring dependencies

Kaggle supplies a CUDA-enabled PyTorch build. Keep it and install the pinned scoring libraries without replacing Torch. Internet must be enabled.


In [ ]:
%pip install -q numpy==1.26.4 transformers==4.40.0 tokenizers==0.19.1 accelerate==0.26.1 peft==0.4.0 bitsandbytes==0.43.0 timm==1.0.3 decord==0.6.0 einops==0.6.1 einops-exts==0.0.4 sentencepiece==0.1.99 tqdm==4.67.1 opencv-python-headless==4.9.0.80 huggingface-hub==0.23.4 safetensors==0.5.2


In [ ]:
# Pin VideoLLaMA2 to the revision used by this repo, apply its low-memory fix, and install without deps.
vlm_src = Path('/kaggle/working/VideoLLaMA2')
if not vlm_src.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DAMO-NLP-SG/VideoLLaMA2.git', str(vlm_src)], check=True)
subprocess.run(['git', '-C', str(vlm_src), 'checkout', 'c0bb03abf6b8a6b9a8dccac006fb4db5d4d9e414'], check=True)
subprocess.run([sys.executable, str(project / 'scripts/patch_videollama2.py'), str(vlm_src)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(vlm_src)], check=True)


In [ ]:
# Download the frozen official checkpoint. This is large; persist it as a Kaggle Model/Dataset if rerunning.
from huggingface_hub import snapshot_download
model_dir = Path('/kaggle/temp/VideoLLaMA2.1-7B-16F')
model_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='DAMO-NLP-SG/VideoLLaMA2.1-7B-16F', local_dir=str(model_dir))
print('Model files:', sum(p.stat().st_size for p in model_dir.rglob('*') if p.is_file()) / 2**30, 'GiB')


## Segment the sample and score segments

For this smoke run, ffmpeg makes non-overlapping 30-second clips. Scoring does not consume the frame bounds as temporal crops; each manifest row must point to its own clip file. Two worker processes each see one physical GPU as `cuda:0`, matching this repo’s current scorer.


In [ ]:
import subprocess, math
work = Path('/kaggle/working/eventvad_run')
clips_dir = work / 'clips'; clips_dir.mkdir(parents=True, exist_ok=True)
# Probe duration using ffprobe (available in Kaggle images).
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(video)], capture_output=True, text=True, check=True)
duration = float(probe.stdout.strip())
window_s = 30
clips = []
for idx, start in enumerate(range(0, math.ceil(duration), window_s)):
    out = clips_dir / f'clip_{idx:04d}.mp4'
    subprocess.run(['ffmpeg','-y','-ss',str(start),'-i',str(video),'-t',str(window_s),'-an','-c:v','libx264','-preset','veryfast','-crf','23','-movflags','+faststart',str(out)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    if out.stat().st_size:
        clips.append((out, start, min(start + window_s, int(duration))))
print(f'{duration:.1f}s input -> {len(clips)} scoring clips')


In [ ]:
manifests = []
for gpu in range(2):
    manifest = work / f'manifest_gpu{gpu}.txt'
    with manifest.open('w') as f:
        for clip, start, end in clips[gpu::2]:
            f.write(f'{clip} {start} {end}\n')
    manifests.append(manifest)

workers = []
for gpu, manifest in enumerate(manifests):
    if not manifest.read_text().strip():
        continue
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)  # scorer's cuda:0 maps to this physical T4
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'backend:cudaMallocAsync'
    env['VIDEOLLAMA2_VISION_BATCH'] = '1'
    output = work / f'scores_gpu{gpu}.txt'
    log = (work / f'worker_gpu{gpu}.log').open('w')
    proc = subprocess.Popen([sys.executable, str(project/'src/score/event_score.py'), '--input_csv', str(manifest), '--output_csv', str(output), '--model', str(model_dir), '--quantization', '4bit'], cwd=project, env=env, stdout=log, stderr=subprocess.STDOUT)
    workers.append((gpu, proc, output, log))

for gpu, proc, output, log in workers:
    code = proc.wait(); log.close()
    print(f'GPU {gpu}: exit={code}; log={work/f"worker_gpu{gpu}.log"}')
    if code:
        print((work/f'worker_gpu{gpu}.log').read_text()[-8000:])
        raise RuntimeError(f'GPU {gpu} scoring process failed')


In [ ]:
# Merge score manifests and detailed raw VLM responses.
merged = work / 'scores.txt'
rows = []
for gpu in range(2):
    part = work / f'scores_gpu{gpu}.txt'
    if part.exists(): rows.extend(line for line in part.read_text().splitlines() if line.strip())
merged.write_text('\n'.join(rows) + ('\n' if rows else ''))
print(merged.read_text())
print('Detailed responses:', *[str(p) for p in sorted(work.glob('scores_gpu*.txt.details.csv'))], sep='\n')


## Outputs and scope

- `eventvad_run/scores.txt`: `clip_path start_sec end_sec anomaly_score` rows.
- `scores_gpu*.txt.details.csv`: raw VLM response and per-clip errors.
- `worker_gpu*.log`: worker logs.

A score of `-1` indicates an inference/parse failure. The fixed windows are only for an end-to-end smoke test; use the repository’s RAFT/CLIP event segmentation stage and official datasets for faithful EventVAD reproduction. The two-GPU arrangement parallelizes clips; it does not combine VRAM.
